# Replication: IISE PG&E Energy Analytics Challenge 2025
## *Hourly-Binned Regression Models Beat Transformers in Load Forecasting*
**Authors:** Millend Roy, Vladimir Pyltsov, Yinbo Hu (Columbia University)

**Replication Scope:** 24 Hourly-Binned XGBoost Regression Models with PCA Exogenous Feature Engineering and Lag/Lead Experiments.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import optuna

from config import TEMP_COLS, GHI_COLS, TARGET_COL, OUTPUT_DIR, FIGURES_DIR
from data_loader import load_and_preprocess_data
from feature_engineering import (
    compute_vif, PCAExogenousTransformer,
    add_all_lag_lead_configurations, get_feature_list_by_config
)
from model_hourly_xgboost import HourlyXGBoostForecaster
from evaluate import (
    compute_metrics, evaluate_year1_to_year2,
    evaluate_year2_to_year1, evaluate_both_years_cv
)
from visualize import (
    plot_fig1_fig2_load_vs_temp, plot_fig3_pca_variance,
    plot_fig4_weekdays_vs_weekends, plot_fig5_xgboost_test_cases,
    plot_fig6_final_forecast, plot_fig7_monthly_forecast
)

print("Environment initialized successfully.")

c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment initialized successfully.


### 1. Load Data & Calendar Feature Reconstruction
- Year 1: 2020 (Leap year starting on Wednesday Jan 1)
- Year 2: 2021
- Year 3: 2022 (Test Set)
- Adds 12 monthly dummies, weekend indicator, and US federal holiday indicator.

In [2]:
train_df, test_df = load_and_preprocess_data(use_cache=True)
print(f"Train shape: {train_df.shape}")
print(f"Test shape : {test_df.shape}")
train_df.head(3)

Train shape: (17544, 32)
Test shape : (8760, 32)


,Year,Month,Day,Hour,Load,Site-1 Temp,Site-2 Temp,Site-3 Temp,Site-4 Temp,Site-5 Temp,...,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9,Month_10,Month_11,Month_12
0,1,1,1,1,1997,8.0,8.2,5.3,9.4,8.1,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,2,1921,8.3,8.6,5.2,8.6,7.1,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,3,1861,8.1,8.8,5.1,8.7,6.2,...,0,0,0,0,0,0,0,0,0,0


### 2. Descriptive Statistics (Table 1 in Paper)

In [3]:
cols = ['Load'] + TEMP_COLS + GHI_COLS
stats_records = []
for yr in [1, 2]:
    sub = train_df[train_df['Year'] == yr]
    for name, func in [('Mean (mu)', 'mean'), ('Std (sigma)', 'std'), ('Median', 'median'), ('Min', 'min'), ('Max', 'max'), ('Skew (gamma1)', 'skew'), ('Kurt (gamma2)', 'kurt')]:
        row = {'Metric': name, 'Year': f'Year {yr}'}
        for c in cols:
            val = getattr(sub[c], func)()
            row[c] = round(float(val), 2)
        stats_records.append(row)
pd.DataFrame(stats_records)

,Metric,Year,Load,Site-1 Temp,Site-2 Temp,Site-3 Temp,Site-4 Temp,Site-5 Temp,Site-1 GHI,Site-2 GHI,Site-3 GHI,Site-4 GHI,Site-5 GHI
0,Mean (mu),Year 1,2162.82,17.31,17.17,18.27,17.72,17.60,225.80,222.47,226.91,227.06,228.39
1,Std (sigma),Year 1,465.77,4.86,4.51,7.36,4.75,5.84,305.18,301.51,307.34,306.63,308.05
2,Median,Year 1,2072.00,17.50,17.40,17.30,17.80,17.30,12.00,12.00,12.00,12.00,13.00
3,Min,Year 1,1101.00,1.90,2.90,-0.50,2.60,0.90,0.00,0.00,0.00,0.00,0.00
4,Max,Year 1,4397.00,36.60,31.90,43.00,36.60,39.70,1037.00,1028.00,1041.00,1047.00,1049.00
5,Skew (gamma1),Year 1,1.18,0.09,-0.00,0.45,0.20,0.42,1.07,1.09,1.09,1.08,1.08
6,Kurt (gamma2),Year 1,2.10,-0.14,-0.33,-0.26,-0.02,0.02,-0.25,-0.19,-0.20,-0.23,-0.23
7,Mean (mu),Year 2,2145.42,16.72,16.47,17.80,17.06,16.84,221.24,218.81,225.35,223.31,225.08
8,Std (sigma),Year 2,406.48,4.48,4.18,7.02,4.35,5.37,301.69,298.73,306.39,304.56,306.06
9,Median,Year 2,2096.00,16.60,16.40,17.20,17.00,16.60,12.00,11.00,11.00,11.00,11.00


### 3. PCA Dimensionality Reduction & VIF Multicollinearity Analysis
- Temperature (5 sites) -> `PCA_Temp` (>97% variance)
- GHI (5 sites) -> `PCA_GHI` (>99% variance)
- Replicates Table 2 & Table 3.

In [4]:
pca_trans = PCAExogenousTransformer()
train_df = pca_trans.fit_transform(train_df)
test_df = pca_trans.transform(test_df)
print("Explained Variance:", pca_trans.get_explained_variance())

print("=== Table 2: VIF Before PCA (Temperature) ===")
display(compute_vif(train_df, TEMP_COLS + [TARGET_COL]))

print("=== Table 2: VIF Before PCA (GHI) ===")
display(compute_vif(train_df, GHI_COLS + [TARGET_COL]))

print("=== Table 3: VIF After PCA ===")
display(compute_vif(train_df, ['PCA_Temp', 'PCA_GHI', TARGET_COL]))

Explained Variance: {'PCA_Temp_Ratio': 0.9705415781814088, 'PCA_GHI_Ratio': 0.9938921177126286}
=== Table 2: VIF Before PCA (Temperature) ===


,Feature,VIF
0,Site-1 Temp,31.736557
1,Site-2 Temp,29.123277
2,Site-3 Temp,17.078576
3,Site-4 Temp,52.697708
4,Site-5 Temp,48.783398
5,Load,1.198797


=== Table 2: VIF Before PCA (GHI) ===


,Feature,VIF
0,Site-1 GHI,180.979002
1,Site-2 GHI,122.906976
2,Site-3 GHI,93.425469
3,Site-4 GHI,692.161170
4,Site-5 GHI,1055.697249
5,Load,1.006675


=== Table 3: VIF After PCA ===


,Feature,VIF
0,PCA_Temp,2.466409
1,PCA_GHI,2.067723
2,Load,1.489578


### 4. Exploratory Visualizations (Figures 1, 2, 3, 4)

In [ ]:
plot_fig1_fig2_load_vs_temp(train_df, FIGURES_DIR)
plot_fig3_pca_variance(train_df, TEMP_COLS, GHI_COLS, FIGURES_DIR)
plot_fig4_weekdays_vs_weekends(train_df, FIGURES_DIR)

### 5. Exogenous Lag & Lead Feature Engineering (Table 5)

In [5]:
train_df = add_all_lag_lead_configurations(train_df)
test_df = add_all_lag_lead_configurations(test_df)
print("Augmented Features Count:", len(train_df.columns))
train_df[['Date', 'Hour', 'PCA_Temp', 'PCA_Temp_Lag1', 'PCA_GHI', 'PCA_GHI_Lag1']].head(3)

Augmented Features Count: 46


,Date,Hour,PCA_Temp,PCA_Temp_Lag1,PCA_GHI,PCA_GHI_Lag1
0,2020-01-01,1,-21.507930,-21.507930,-501.898296,-501.898296
1,2020-01-01,2,-22.080623,-21.507930,-501.898296,-501.898296
2,2020-01-01,3,-22.530378,-22.080623,-501.898296,-501.898296


### 6. XGBoost 24-Hourly Model Experiments (Table 4 & Table 5)

In [6]:
from run_experiments import run_lag_lead_experiments
table5_df = run_lag_lead_experiments(train_df, n_tune_trials=10)
table5_df


 TABLE 5: XGBOOST IMPROVEMENT RESULTS (LAG AND LEAD EXPERIMENTS)
Loading existing Table 5 results from: d:\AIO2026\M03\output\results_table5_xgboost_improvements.csv
Model_Config  Y1_Y2_R2  Y1_Y2_RMSE  Y1_Y2_MAPE  Y1_Y2_sMAPE  Y2_Y1_R2  Y2_Y1_RMSE  Y2_Y1_MAPE  Y2_Y1_sMAPE  Both_R2  Both_RMSE  Both_MAPE  Both_sMAPE
    Baseline      0.84      160.16        5.43         5.44      0.85      178.01        5.61         5.57     0.67     257.87       6.87        7.21
        Lag1      0.84      160.48        5.42         5.44      0.85      178.60        5.66         5.61     0.67     255.59       6.85        7.16
        Lag2      0.85      158.94        5.35         5.39      0.85      179.69        5.69         5.63     0.68     253.64       6.79        7.10
       Lead1      0.84      161.12        5.38         5.40      0.85      178.75        5.63         5.59     0.68     254.53       6.79        7.10
        Lag5      0.86      149.89        5.15         5.20      0.87      168.76  

,Model_Config,Y1_Y2_R2,Y1_Y2_RMSE,Y1_Y2_MAPE,Y1_Y2_sMAPE,Y2_Y1_R2,Y2_Y1_RMSE,Y2_Y1_MAPE,Y2_Y1_sMAPE,Both_R2,Both_RMSE,Both_MAPE,Both_sMAPE
0,Baseline,0.84,160.16,5.43,5.44,0.85,178.01,5.61,5.57,0.67,257.87,6.87,7.21
1,Lag1,0.84,160.48,5.42,5.44,0.85,178.60,5.66,5.61,0.67,255.59,6.85,7.16
2,Lag2,0.85,158.94,5.35,5.39,0.85,179.69,5.69,5.63,0.68,253.64,6.79,7.10
3,Lead1,0.84,161.12,5.38,5.40,0.85,178.75,5.63,5.59,0.68,254.53,6.79,7.10
4,Lag5,0.86,149.89,5.15,5.20,0.87,168.76,5.60,5.51,0.69,250.20,6.67,7.00
5,Lag1+Lead1,0.85,159.60,5.35,5.38,0.85,179.97,5.66,5.61,0.68,253.20,6.75,7.06
6,Lag3,0.85,155.36,5.27,5.30,0.86,177.25,5.67,5.60,0.68,251.50,6.72,7.04
7,Lag2+Lead2,0.85,157.70,5.31,5.35,0.85,177.90,5.63,5.57,0.69,251.17,6.69,7.01
8,Lead2,0.84,160.85,5.41,5.43,0.85,177.52,5.64,5.59,0.68,255.14,6.83,7.15
9,Lag3+Lead2,0.85,155.28,5.26,5.30,0.86,175.38,5.62,5.55,0.69,250.04,6.65,6.98


### 7. Figure 5: XGBoost Model Test Cases

In [7]:
final_features = get_feature_list_by_config('Lag1')
_, y2_preds = evaluate_year1_to_year2(train_df, final_features, tune_trials=10)
_, y1_preds = evaluate_year2_to_year1(train_df, final_features, tune_trials=10)
_, cv_preds, _ = evaluate_both_years_cv(train_df, final_features, n_splits=5, tune_trials=0)

y1_df = train_df[train_df['Year'] == 1]
y2_df = train_df[train_df['Year'] == 2]
df_sorted = train_df.sort_values(['Year', 'Month', 'Day', 'Hour']).reset_index(drop=True)
n_cv = len(cv_preds)
cv_trues = df_sorted.iloc[-n_cv:]['Load'].values

plot_fig5_xgboost_test_cases(y1_df, y2_df, y2_preds, y1_preds, cv_trues, cv_preds, FIGURES_DIR)

Saved: d:\AIO2026\M03\output\figures\Figure_5_Test_Cases_XGBoost.png


### 8. Train Final Model on Full 2-Year Dataset & Forecast Year 3 (Figures 6 & 7)

In [8]:
from run_experiments import train_final_model_and_forecast
y3_preds, forecaster = train_final_model_and_forecast(train_df, test_df, final_features, n_tune_trials=15)
plot_fig6_final_forecast(test_df, y3_preds, FIGURES_DIR)
plot_fig7_monthly_forecast(test_df, y3_preds, FIGURES_DIR)


 TRAINING FINAL XGBOOST MODEL (LAG1 CONFIGURATION) ON FULL 2-YEAR DATASET
Starting Optuna hyperparameter optimization across 24 hourly models (n_trials=15)...
  Hour  1/24 optimized: depth=3, n_est=329, lr=0.0257
  Hour  6/24 optimized: depth=4, n_est=296, lr=0.0421
  Hour 12/24 optimized: depth=5, n_est=295, lr=0.0339
  Hour 18/24 optimized: depth=6, n_est=338, lr=0.0321
  Hour 24/24 optimized: depth=3, n_est=172, lr=0.0386

Generating Year 3 (Test Set) load predictions...

 YEAR 3 (TEST DATASET) EVALUATION METRICS
  R2   : 0.8762
  RMSE : 169.74
  MAPE : 5.37%
  sMAPE: 5.45%

Final predictions successfully exported to:
  - d:\AIO2026\M03\output\predictions_xgboost.xlsx
  - d:\AIO2026\M03\output\predictions_xgboost.csv

Top 10 Important Features across 24 Hourly Models:
      Feature  Importance
PCA_Temp_Lag1    0.147035
   Is_Weekend    0.117004
     PCA_Temp    0.114649
      Month_4    0.085618
      Month_8    0.077910
     Month_12    0.077523
      Month_5    0.071687
      Mon